In [1]:
!pip install pyspark -q

In [14]:
import os
import zipfile
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lpad, floor

outer_zip_path = "/content/BTS_Data_2025.zip"
outer_extract_dir = "/content/bts_data"
os.makedirs(outer_extract_dir, exist_ok=True)

# Extract outer zip
with zipfile.ZipFile(outer_zip_path, 'r') as zip_ref:
    zip_ref.extractall(outer_extract_dir)

print("Outer zip extracted.")
print("Contents of outer_extract_dir:")
for item in os.listdir(outer_extract_dir):
    print(item)

Outer zip extracted.
Contents of outer_extract_dir:
BTS_Data_2025
extracted_monthly


In [6]:
for root, dirs, files in os.walk(outer_extract_dir):
    print("Folder:", root)
    print("Files:", files)
    print("-----")

Folder: /content/bts_data
Files: []
-----
Folder: /content/bts_data/BTS_Data_2025
Files: ['On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_12.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_6.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_9.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_2.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_10.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_4.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_3.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_5.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_7.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_11.zip', 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_8.zip', 'On_Time_Reporting_Carrier_On_T

In [7]:
# Folder that contains the monthly zip files
monthly_zip_dir = "/content/bts_data/BTS_Data_2025"

# Folder to extract all monthly zips
monthly_extract_dir = "/content/bts_data/extracted_monthly"
os.makedirs(monthly_extract_dir, exist_ok=True)

# Collect all inner zip files, and ignore duplicate files like "... 2.zip"
monthly_zip_files = [
    f for f in os.listdir(monthly_zip_dir)
    if f.endswith(".zip") and " 2" not in f
]

print("Monthly zip files to process:", len(monthly_zip_files))
for f in monthly_zip_files:
    print(f)

# Extract each monthly zip into its own subfolder
for zip_file in monthly_zip_files:
    zip_path = os.path.join(monthly_zip_dir, zip_file)

    # Use zip filename (without extension) as subfolder name
    folder_name = os.path.splitext(zip_file)[0].replace(" ", "_")
    target_dir = os.path.join(monthly_extract_dir, folder_name)
    os.makedirs(target_dir, exist_ok=True)

    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(target_dir)
        print(f"Extracted: {zip_file}")
    except Exception as e:
        print(f"Failed to extract {zip_file}: {e}")

Monthly zip files to process: 12
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_12.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_6.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_9.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_2.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_10.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_4.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_3.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_5.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_7.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_11.zip
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_8.zip
Extracted: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_12.zip
Extracted: On_Time_Reporting_Carrier_On_Ti

In [8]:
csv_files = []

for root, dirs, files in os.walk(monthly_extract_dir):
    for file in files:
        if file.endswith(".csv"):
            csv_files.append(os.path.join(root, file))

# Sort paths so they are in a stable order
csv_files = sorted(csv_files)

print("Number of CSV files found:", len(csv_files))
print("First few CSV files:")
for f in csv_files[:5]:
    print(f)

Number of CSV files found: 24
First few CSV files:
/content/bts_data/extracted_monthly/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv
/content/bts_data/extracted_monthly/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1/__MACOSX/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1/._On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv
/content/bts_data/extracted_monthly/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_10/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_10/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_10.csv
/content/bts_data/extracted_monthly/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_10/__MACOSX/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_10/._On_Time_Reporting_Carrier_On_Time_

In [10]:
spark = SparkSession.builder \
    .appName("BTS_Airline_Delay_Cleaning") \
    .getOrCreate()

print("Spark session is ready.")

Spark session is ready.


In [12]:
df = spark.read.csv(
    csv_files,
    header=True,
    inferSchema=True
)

# Preview data
df.show(5, truncate=False)

# Print schema
df.printSchema()

# Check dataset size
print("Row count:", df.count())
print("Column count:", len(df.columns))
print("All columns:")
for c in df.columns:
    print(c)

+----+-------+-----+----------+---------+----------+-----------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+------+--------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+----+------------+---------+-------------+-------------+-------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+----------+-------+--------+---------------+--------+------------------+----------+---------+----------------+--------+--------------+-----------------+-------+-------+--------+-------------+------------+------------+--------+-------------+-----------------+------------+-------------+---------------+------------------+--------------+--------------------+-----------+-----------+-----------+-------------+----------------+------------+--------------+----------------+----

In [15]:
core_columns = [
    "Year", "Quarter", "Month", "DayofMonth", "DayOfWeek",
    "FlightDate",

    "Reporting_Airline", "DOT_ID_Reporting_Airline", "IATA_CODE_Reporting_Airline",

    "Origin", "OriginCityName", "OriginState",
    "Dest", "DestCityName", "DestState",

    "CRSDepTime", "DepTime", "DepDelay", "DepDelayMinutes",
    "CRSArrTime", "ArrTime", "ArrDelay", "ArrDelayMinutes",

    "Cancelled", "CancellationCode", "Diverted",

    "CRSElapsedTime", "ActualElapsedTime", "AirTime", "Distance",

    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay"
]

df_core = df.select(*core_columns)

print("Selected columns:", len(df_core.columns))
df_core.show(5, truncate=False)

Selected columns: 35
+----+-------+-----+----------+---------+----------+-----------------+------------------------+---------------------------+------+--------------+-----------+----+------------+---------+----------+-------+--------+---------------+----------+-------+--------+---------------+---------+----------------+--------+--------------+-----------------+-------+--------+------------+------------+--------+-------------+-----------------+
|Year|Quarter|Month|DayofMonth|DayOfWeek|FlightDate|Reporting_Airline|DOT_ID_Reporting_Airline|IATA_CODE_Reporting_Airline|Origin|OriginCityName|OriginState|Dest|DestCityName|DestState|CRSDepTime|DepTime|DepDelay|DepDelayMinutes|CRSArrTime|ArrTime|ArrDelay|ArrDelayMinutes|Cancelled|CancellationCode|Diverted|CRSElapsedTime|ActualElapsedTime|AirTime|Distance|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|
+----+-------+-----+----------+---------+----------+-----------------+------------------------+---------------------------+--

In [16]:
df_clean = df_core \
    .withColumn("is_cancelled", col("Cancelled").cast("int")) \
    .withColumn("is_diverted", col("Diverted").cast("int")) \
    .withColumn("is_dep_delayed", when(col("DepDelayMinutes") > 15, 1).otherwise(0)) \
    .withColumn("is_arr_delayed", when(col("ArrDelayMinutes") > 15, 1).otherwise(0)) \
    .withColumn("CRSDepTime_PADDED", lpad(col("CRSDepTime").cast("string"), 4, "0")) \
    .withColumn("dep_hour", floor(col("CRSDepTime_PADDED").substr(1, 2).cast("int"))) \
    .withColumn("season", when(col("Month").isin(12, 1, 2), "Winter")
        .when(col("Month").isin(3, 4, 5), "Spring")
        .when(col("Month").isin(6, 7, 8), "Summer")
        .otherwise("Fall")
    )

In [17]:
delay_cols = [
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay"
]

for c in delay_cols:
    df_clean = df_clean.withColumn(
        c,
        when(col(c).isNull(), 0).otherwise(col(c))
    )

print("Delay column clean done.")

Delay columns cleaned.


In [18]:
df_clean.createOrReplaceTempView("airline_data")
print("SQL table 'airline_data' created.")

SQL table 'airline_data' created.


In [19]:
df_clean.select(
    "Year", "Month", "DayOfWeek",
    "Reporting_Airline",
    "Origin", "Dest",
    "CRSDepTime", "dep_hour",
    "DepDelayMinutes", "ArrDelayMinutes",
    "is_dep_delayed", "is_arr_delayed",
    "is_cancelled", "is_diverted",
    "season"
).show(10, False)

+----+-----+---------+-----------------+------+----+----------+--------+---------------+---------------+--------------+--------------+------------+-----------+------+
|Year|Month|DayOfWeek|Reporting_Airline|Origin|Dest|CRSDepTime|dep_hour|DepDelayMinutes|ArrDelayMinutes|is_dep_delayed|is_arr_delayed|is_cancelled|is_diverted|season|
+----+-----+---------+-----------------+------+----+----------+--------+---------------+---------------+--------------+--------------+------------+-----------+------+
|2024|1    |1        |9E               |LGA   |OMA |856       |8       |0.0            |0.0            |0             |0             |0           |0          |Winter|
|2024|1    |2        |9E               |LGA   |OMA |856       |8       |0.0            |0.0            |0             |0             |0           |0          |Winter|
|2024|1    |3        |9E               |LGA   |OMA |856       |8       |0.0            |0.0            |0             |0             |0           |0          |Winter